# v7 Python Module API Quickstart

This notebook shows the thin `ck.models` / `ck.nn` authoring surface on top of the existing `v7` training pipeline.
It does not replace v7 lowering or runtime execution; it generates a symbolic module graph, then hands off to the same `ck_run_v7.py` path used by the earlier Python authoring notebooks.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "ckernel_engine").exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import ckernel_engine as ck
from IPython.display import HTML, Markdown, display


In [ ]:
model = ck.models.qwen3_tiny(
    vocab=256,
    dim=128,
    layers=2,
    hidden=256,
    heads=8,
    kv_heads=4,
    context_len=128,
    init="xavier_uniform",
    name="tiny_qwen3_module_api",
)

run = ck.v7.compile(
    model,
    run_name="python-module-api-demo",
    family="qwen3",
    init="xavier_uniform",
    config=ck.CompileConfig(
        target=ck.TargetConfig(name="cpu", isa="auto"),
        vectorize=True,
        pack_weights=True,
        dump_pass_trace=True,
    ),
)
display(Markdown(run.show_graph()))


The next cell is the same old v7 flow: materialize the run, train on inline text, refresh the IR report, and rebuild the shared run hub.


In [ ]:
materialize_result = run.materialize()
train_result = run.train(
    "C-Kernel-Engine module API notebook quickstart.",
    ck.v7.TrainConfig(
        backend="ck",
        strict=True,
        epochs=1,
        seq_len=8,
        total_tokens=64,
        grad_accum=2,
        lr=5e-4,
        parity_regimen="suggest",
        memory_check=False,
    ),
)
viewer_artifacts = run.prepare_viewers()

print(materialize_result.run_dir)
print(train_result.report_path)
print(viewer_artifacts.ir_report)
print(viewer_artifacts.ir_hub)


In [ ]:
display(run.show_compile_config())
display(run.show_pass_trace())
display(HTML(run.notebook_artifact_dashboard_html(title="Module API Artifact Dashboard")))
